In [2]:
import mlflow

mlflow.set_tracking_uri(
    "https://dagshub.com/Tostaza/MLOps-Datatouille.mlflow"
)

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: https://dagshub.com/Tostaza/MLOps-Datatouille.mlflow


In [ ]:
from mlflow import MlflowClient

RUN_ID = "3b92770faa25418792e2f2ce1f387102"

client = MlflowClient()                         # OBRIM UN CLIENT MLFLOW PER ANAR A FER CONSULTES A LA BASE DE DADES DE MLFLOW.

run = client.get_run(RUN_ID)

print("Run ID:", run.info.run_id)
print("Status:", run.info.status)
print("Artifact URI:", run.info.artifact_uri)

Run ID: 3b92770faa25418792e2f2ce1f387102
Status: FINISHED
Artifact URI: mlflow-artifacts:/57ea5fbda5d14d6b9dfcb1c8387a983b/3b92770faa25418792e2f2ce1f387102/artifacts


In [5]:
artifacts = client.list_artifacts(RUN_ID)

for artifact in artifacts:
    print(
        f"Path: {artifact.path} | "
        f"Directory: {artifact.is_dir}"
    )

#### Let's search the model associated with the run_id proposed above

In [6]:
import mlflow

models = mlflow.search_logged_models(
    experiment_ids=[run.info.experiment_id],
    filter_string=f"source_run_id = '{RUN_ID}'",
    output_format="list"
)

print("Models trobats:", len(models))

for model in models:
    print("Nom:", model.name)
    print("Model ID:", model.model_id)
    print("Model URI:", model.model_uri)

Models trobats: 0


In [ ]:
token_hugging_face = ""

### API HUGGING FACE FOR MODEL SUMMARIZATION 

In [10]:
import requests

def resumir_con_token_gratuito(texto_largo):
    url = "https://huggingface.co"
    
    # Reemplaza 'hf_...' con el token de lectura que has copiado de tu cuenta
    token = "hf_TU_TOKEN_REAL_AQUI"
    
    cabeceras = {
        "Authorization": f"Bearer {token}",
        "User-Agent": "Mozilla/5.0"
    }
    
    payload = {
        "inputs": texto_largo,
        "parameters": {
            "max_length": 150,
            "min_length": 40,
            "do_sample": False
        }
    }
    
    response = requests.post(url, json=payload, headers=cabeceras)
    
    if response.status_code == 200:
        # Extraemos el resumen del primer elemento de la lista devuelta
        return response.json()[0]['summary_text']
    elif response.status_code == 503:
        return "El modelo se está despertando en los servidores de Hugging Face. Espera 15 segundos y vuelve a ejecutar."
    else:
        return f"Error {response.status_code}: {response.text}"

# --- EJEMPLO DE PRUEBA ---
texto_prueba = """
El telescopio espacial James Webb ha revelado imágenes sin precedentes de galaxias lejanas. 
Los astrónomos confirman que los datos obtenidos permitirán reescribir las teorías actuales 
sobre la formación del universo primitivo. Este proyecto ha tomado más de dos décadas de desarrollo, 
pero los primeros resultados demuestran que ha valido la pena de forma absoluta.
"""

print("Solicitando resumen con autenticación...")
resultado = resumir_con_token_gratuito(texto_prueba)
print("\nResultado final:\n", resultado)


Solicitando resumen con autenticación...

Resultado final:
 Error 403: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
This distribution is not configured to allow the HTTP request method that was used for this request. The distribution supports only cachable requests.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.
<BR clear="all">
If you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.
<BR clear="all">
<HR noshade size="1px">
<PRE>
Generated by cloudfront 